# 03. Fusion diagnostics (read-only)

Цей notebook **не запускає GTSAM повторно і не змінює алгоритм**. Він читає вже створені артефакти `PROJECT_CV_ARTIFACTS/fusion_v1`, порівнює synthetic GPS з dense GPS, аналізує Starlink, висоту, покриття optical-flow velocity та зберігає відтворюваний діагностичний звіт.

Дві оцінки synthetic GPS навмисно розділені:

- `linear_interpolation_offline_noncausal` — offline-оцінка у спільні моменти часу, яка використовує обидва сусідні output-вузли;
- `previous_output_sample_causal_zoh` — останній уже опублікований output-вузол (zero-order hold), без погляду в майбутнє.

Dense GPS тут є лише reference для діагностики. Вхідні CSV не перезаписуються.

In [ ]:
from __future__ import annotations

import json
import os
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

assert platform.system() == 'Linux', (
    'Select the Project CV (ROS 2 Humble) kernel; '
    f'the current kernel reports {platform.system()}.'
)
assert sys.version_info[:2] == (3, 10), platform.python_version()
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 180)
print('Python:', platform.python_version())
print('Mode: read existing fusion artifacts; GTSAM is not imported or executed.')

In [ ]:
def required_env_path(name: str) -> Path:
    value = os.environ.get(name)
    assert value, f'Missing environment variable: {name}'
    return Path(value)


ARTIFACTS_ROOT = required_env_path('PROJECT_CV_ARTIFACTS')
RUN_ROOT = ARTIFACTS_ROOT / 'fusion_v1'
WORK_DIR = RUN_ROOT / 'work'
GTSAM_DIR = RUN_ROOT / 'gtsam'
DIAGNOSTICS_DIR = RUN_ROOT / 'diagnostics'
SUMMARY_PATH = GTSAM_DIR / 'gtsam_incremental_fixedlag_summary.json'
CALIBRATION_PATH = WORK_DIR / 'calibration.json'
GPS_PATH = WORK_DIR / 'gps_ref_enu.csv'
STARLINK_PATH = WORK_DIR / 'starlink_corrected_enu.csv'
VISION_PATH = WORK_DIR / 'vision_velocity_corrected_frd.csv'
BARO_PATH = WORK_DIR / 'baro_corrected.csv'
GPS_IMU_SCAN_PATH = WORK_DIR / 'gps_imu_delay_scan.csv'
STARLINK_SCAN_PATH = WORK_DIR / 'delay_scan.csv'
required_inputs = [SUMMARY_PATH, CALIBRATION_PATH, GPS_PATH, STARLINK_PATH, VISION_PATH, BARO_PATH, GPS_IMU_SCAN_PATH, STARLINK_SCAN_PATH]
missing = [str(path) for path in required_inputs if not path.exists()]
assert not missing, f'Missing fusion_v1 artifacts. Run 02_sparse_gps_fusion.ipynb first: {missing}'
summary: dict[str, Any] = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
calibration: dict[str, Any] = json.loads(CALIBRATION_PATH.read_text(encoding='utf-8'))
output_hz = float(summary.get('output_hz', 5.0))
preferred_output = GTSAM_DIR / f'synthetic_gps_{int(round(output_hz))}hz_gtsam_fixedlag.csv'
if preferred_output.exists():
    OUTPUT_PATH = preferred_output
else:
    candidates = sorted(GTSAM_DIR.glob('synthetic_gps_*hz_gtsam_fixedlag.csv'))
    assert len(candidates) == 1, f'Cannot select one synthetic GPS output: {candidates}'
    OUTPUT_PATH = candidates[0]
DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)
print('Fusion artifacts :', RUN_ROOT)
print('Synthetic output :', OUTPUT_PATH)
print('Diagnostics only :', DIAGNOSTICS_DIR)

In [ ]:
def read_csv_checked(path: Path, required_columns: set[str]) -> pd.DataFrame:
    frame = pd.read_csv(path)
    missing_columns = sorted(required_columns - set(frame.columns))
    if missing_columns:
        raise ValueError(f'{path.name}: missing columns {missing_columns}')
    return frame


def finite_sorted(frame: pd.DataFrame, time_column: str, value_columns: list[str]) -> pd.DataFrame:
    result = frame.copy()
    numeric_columns = [time_column, *value_columns]
    for column in numeric_columns:
        result[column] = pd.to_numeric(result[column], errors='coerce')
    mask = np.ones(len(result), dtype=bool)
    for column in numeric_columns:
        mask &= np.isfinite(result[column].to_numpy(dtype=float))
    return result.loc[mask].sort_values(time_column).drop_duplicates(subset=[time_column], keep='last').reset_index(drop=True)


estimate = finite_sorted(read_csv_checked(OUTPUT_PATH, {'tr', 'e', 'n', 'u'}), 'tr', ['e', 'n', 'u'])
gps = finite_sorted(read_csv_checked(GPS_PATH, {'tr', 'e', 'n', 'u'}), 'tr', ['e', 'n', 'u'])
starlink = finite_sorted(read_csv_checked(STARLINK_PATH, {'tr_corr', 'e_corr', 'n_corr', 'u_corr'}), 'tr_corr', ['e_corr', 'n_corr', 'u_corr'])
vision = finite_sorted(read_csv_checked(VISION_PATH, {'tr_corr', 'vx', 'vy', 'vz'}), 'tr_corr', ['vx', 'vy', 'vz'])
barometer = finite_sorted(read_csv_checked(BARO_PATH, {'tr', 'u_corr'}), 'tr', ['u_corr'])
gps_imu_scan = pd.read_csv(GPS_IMU_SCAN_PATH)
starlink_scan = pd.read_csv(STARLINK_SCAN_PATH)
assert len(estimate) >= 2 and len(gps) >= 2, 'Not enough estimate/reference rows'
input_overview = pd.DataFrame([
    {'dataset': 'synthetic_gps', 'rows': len(estimate), 't_min': estimate.tr.min(), 't_max': estimate.tr.max()},
    {'dataset': 'dense_gps_reference', 'rows': len(gps), 't_min': gps.tr.min(), 't_max': gps.tr.max()},
    {'dataset': 'starlink_corrected', 'rows': len(starlink), 't_min': starlink.tr_corr.min(), 't_max': starlink.tr_corr.max()},
    {'dataset': 'vision_velocity_corrected', 'rows': len(vision), 't_min': vision.tr_corr.min(), 't_max': vision.tr_corr.max()},
    {'dataset': 'barometer_corrected', 'rows': len(barometer), 't_min': barometer.tr.min(), 't_max': barometer.tr.max()},
])
display(input_overview)

## Synthetic output проти dense GPS

Інтерполяційна метрика відповідає на питання «де проходить оцінена траєкторія в момент reference-виміру?». Causal previous-sample метрика відповідає на питання «яке останнє доступне navigation-рішення було б опубліковано в цей момент?». Їх не можна змішувати в одне число.

In [ ]:
INTERPOLATED_MODE = 'linear_interpolation_offline_noncausal'
CAUSAL_MODE = 'previous_output_sample_causal_zoh'


def add_position_errors(table: pd.DataFrame) -> pd.DataFrame:
    result = table.copy()
    for axis in ('e', 'n', 'u'):
        result[f'err_{axis}'] = result[f'est_{axis}'] - result[f'ref_{axis}']
    result['err_xy'] = np.hypot(result['err_e'], result['err_n'])
    result['err_3d'] = np.sqrt(result['err_e'] ** 2 + result['err_n'] ** 2 + result['err_u'] ** 2)
    result['abs_err_u'] = result['err_u'].abs()
    return result


def align_estimate_to_reference(
    estimate_frame: pd.DataFrame,
    reference_frame: pd.DataFrame,
    mode: str,
    altitude_column: str = 'u',
) -> pd.DataFrame:
    est = finite_sorted(estimate_frame, 'tr', ['e', 'n', altitude_column])
    ref = finite_sorted(reference_frame, 'tr', ['e', 'n', 'u'])
    overlap = (ref['tr'] >= est['tr'].min()) & (ref['tr'] <= est['tr'].max())
    ref = ref.loc[overlap].copy().reset_index(drop=True)
    assert len(ref) >= 2, 'Estimate and reference have no useful overlap'
    est_t = est['tr'].to_numpy(dtype=float)
    ref_t = ref['tr'].to_numpy(dtype=float)
    out = pd.DataFrame({
        'tr': ref_t,
        'ref_e': ref['e'].to_numpy(dtype=float),
        'ref_n': ref['n'].to_numpy(dtype=float),
        'ref_u': ref['u'].to_numpy(dtype=float),
    })
    previous_index = np.searchsorted(est_t, ref_t, side='right') - 1
    valid_previous = previous_index >= 0
    if not np.all(valid_previous):
        out = out.loc[valid_previous].reset_index(drop=True)
        ref_t = ref_t[valid_previous]
        previous_index = previous_index[valid_previous]
    if mode == INTERPOLATED_MODE:
        out['est_e'] = np.interp(ref_t, est_t, est['e'].to_numpy(dtype=float))
        out['est_n'] = np.interp(ref_t, est_t, est['n'].to_numpy(dtype=float))
        out['est_u'] = np.interp(ref_t, est_t, est[altitude_column].to_numpy(dtype=float))
        out['output_sample_age_sec'] = np.nan
    elif mode == CAUSAL_MODE:
        out['est_e'] = est['e'].to_numpy(dtype=float)[previous_index]
        out['est_n'] = est['n'].to_numpy(dtype=float)[previous_index]
        out['est_u'] = est[altitude_column].to_numpy(dtype=float)[previous_index]
        out['output_sample_age_sec'] = ref_t - est_t[previous_index]
    else:
        raise ValueError(f'Unknown matching mode: {mode}')
    out['matching_mode'] = mode
    out['altitude_series'] = altitude_column
    return add_position_errors(out)

In [ ]:
def summarize_position_errors(table: pd.DataFrame, mode: str) -> dict[str, Any]:
    row: dict[str, Any] = {'scope': 'synthetic_gps_vs_dense_gps', 'matching_mode': mode, 'n': int(len(table))}
    for axis in ('e', 'n', 'u'):
        error = table[f'err_{axis}'].to_numpy(dtype=float)
        row[f'bias_{axis}_m'] = float(np.mean(error))
        row[f'rmse_{axis}_m'] = float(np.sqrt(np.mean(error ** 2)))
        row[f'mae_{axis}_m'] = float(np.mean(np.abs(error)))
        row[f'max_abs_{axis}_m'] = float(np.max(np.abs(error)))
    xy = table['err_xy'].to_numpy(dtype=float)
    error_3d = table['err_3d'].to_numpy(dtype=float)
    row.update({
        'rmse_xy_m': float(np.sqrt(np.mean(xy ** 2))),
        'mae_xy_m': float(np.mean(xy)),
        'p95_xy_m': float(np.percentile(xy, 95)),
        'max_xy_m': float(np.max(xy)),
        'rmse_3d_m': float(np.sqrt(np.mean(error_3d ** 2))),
        'p95_abs_u_m': float(np.percentile(table['abs_err_u'], 95)),
    })
    ages = table['output_sample_age_sec'].dropna().to_numpy(dtype=float)
    row['p95_output_sample_age_sec'] = float(np.percentile(ages, 95)) if len(ages) else None
    row['max_output_sample_age_sec'] = float(np.max(ages)) if len(ages) else None
    return row


aligned_interpolated = align_estimate_to_reference(estimate, gps, INTERPOLATED_MODE)
aligned_causal = align_estimate_to_reference(estimate, gps, CAUSAL_MODE)
position_metrics = pd.DataFrame([
    summarize_position_errors(aligned_interpolated, INTERPOLATED_MODE),
    summarize_position_errors(aligned_causal, CAUSAL_MODE),
])
aligned_interpolated.to_csv(DIAGNOSTICS_DIR / 'output_vs_dense_gps_interpolated.csv', index=False)
aligned_causal.to_csv(DIAGNOSTICS_DIR / 'output_vs_dense_gps_causal_previous.csv', index=False)
position_metrics.to_csv(DIAGNOSTICS_DIR / 'output_vs_dense_gps_metrics.csv', index=False)
display(position_metrics)

In [ ]:
def align_altitude_series(
    samples: pd.DataFrame,
    reference: pd.DataFrame,
    sample_time: str,
    sample_value: str,
    mode: str,
    series_name: str,
) -> pd.DataFrame:
    source = finite_sorted(samples, sample_time, [sample_value])
    ref = finite_sorted(reference, 'tr', ['u'])
    overlap = (ref['tr'] >= source[sample_time].min()) & (ref['tr'] <= source[sample_time].max())
    ref = ref.loc[overlap].copy().reset_index(drop=True)
    source_t = source[sample_time].to_numpy(dtype=float)
    ref_t = ref['tr'].to_numpy(dtype=float)
    previous_index = np.searchsorted(source_t, ref_t, side='right') - 1
    valid = previous_index >= 0
    ref = ref.loc[valid].copy().reset_index(drop=True)
    ref_t = ref_t[valid]
    previous_index = previous_index[valid]
    if mode == INTERPOLATED_MODE:
        estimated_u = np.interp(ref_t, source_t, source[sample_value].to_numpy(dtype=float))
        age = np.full(len(ref_t), np.nan)
    elif mode == CAUSAL_MODE:
        estimated_u = source[sample_value].to_numpy(dtype=float)[previous_index]
        age = ref_t - source_t[previous_index]
    else:
        raise ValueError(mode)
    result = pd.DataFrame({
        'tr': ref_t, 'ref_u': ref['u'].to_numpy(dtype=float), 'est_u': estimated_u,
        'sample_age_sec': age, 'matching_mode': mode, 'series': series_name,
    })
    result['err_u'] = result['est_u'] - result['ref_u']
    result['abs_err_u'] = result['err_u'].abs()
    return result


def summarize_altitude(table: pd.DataFrame) -> dict[str, Any]:
    error = table['err_u'].to_numpy(dtype=float)
    ages = table['sample_age_sec'].dropna().to_numpy(dtype=float)
    return {
        'scope': 'altitude_vs_dense_gps',
        'series': str(table['series'].iloc[0]),
        'matching_mode': str(table['matching_mode'].iloc[0]),
        'n': int(len(table)),
        'bias_u_m': float(np.mean(error)),
        'rmse_u_m': float(np.sqrt(np.mean(error ** 2))),
        'mae_u_m': float(np.mean(np.abs(error))),
        'p95_abs_u_m': float(np.percentile(np.abs(error), 95)),
        'max_abs_u_m': float(np.max(np.abs(error))),
        'p95_sample_age_sec': float(np.percentile(ages, 95)) if len(ages) else None,
    }

In [ ]:
altitude_tables: list[pd.DataFrame] = []
for matching_mode in (INTERPOLATED_MODE, CAUSAL_MODE):
    altitude_tables.append(align_altitude_series(estimate, gps, 'tr', 'u', matching_mode, 'fusion_exported_u'))
    if 'u_raw' in estimate.columns:
        altitude_tables.append(align_altitude_series(estimate, gps, 'tr', 'u_raw', matching_mode, 'fusion_raw_u'))
    altitude_tables.append(align_altitude_series(barometer, gps, 'tr', 'u_corr', matching_mode, 'prepared_baro_u_corr'))

altitude_metrics = pd.DataFrame([summarize_altitude(table) for table in altitude_tables])
altitude_details = pd.concat(altitude_tables, ignore_index=True)
altitude_metrics.to_csv(DIAGNOSTICS_DIR / 'altitude_metrics.csv', index=False)
altitude_details.to_csv(DIAGNOSTICS_DIR / 'altitude_alignment_details.csv', index=False)
display(altitude_metrics)

## Starlink проти dense GPS

Reference GPS інтерполюється в corrected Starlink timestamps. `stale` означає: Starlink step ≤ 2 м, але reference step ≥ 10 м. XY-outlier threshold — максимум із 30 м і robust межі `median + 4 × 1.4826 × MAD`.

In [ ]:
STARLINK_STALE_MAX_STEP_M = 2.0
REFERENCE_MOVEMENT_FOR_STALE_M = 10.0
STARLINK_OUTLIER_FLOOR_M = 30.0
STARLINK_ALTITUDE_OUTLIER_FLOOR_M = 20.0
ROBUST_OUTLIER_MULTIPLIER = 4.0


def robust_upper_limit(values: np.ndarray, floor: float) -> tuple[float, float, float]:
    finite = values[np.isfinite(values)]
    median = float(np.median(finite))
    mad = float(np.median(np.abs(finite - median)))
    robust_sigma = 1.4826 * mad
    return max(float(floor), median + ROBUST_OUTLIER_MULTIPLIER * robust_sigma), median, mad


sl = starlink.copy()
in_gps = (sl['tr_corr'] >= gps['tr'].min()) & (sl['tr_corr'] <= gps['tr'].max())
sl = sl.loc[in_gps].copy().reset_index(drop=True)
sl_t = sl['tr_corr'].to_numpy(dtype=float)
gps_t = gps['tr'].to_numpy(dtype=float)
starlink_diagnostics = pd.DataFrame({
    'tr_corr': sl_t,
    'starlink_e': sl['e_corr'].to_numpy(dtype=float),
    'starlink_n': sl['n_corr'].to_numpy(dtype=float),
    'starlink_u': sl['u_corr'].to_numpy(dtype=float),
    'gps_e_interpolated': np.interp(sl_t, gps_t, gps['e'].to_numpy(dtype=float)),
    'gps_n_interpolated': np.interp(sl_t, gps_t, gps['n'].to_numpy(dtype=float)),
    'gps_u_interpolated': np.interp(sl_t, gps_t, gps['u'].to_numpy(dtype=float)),
})
starlink_diagnostics['err_e'] = starlink_diagnostics['starlink_e'] - starlink_diagnostics['gps_e_interpolated']
starlink_diagnostics['err_n'] = starlink_diagnostics['starlink_n'] - starlink_diagnostics['gps_n_interpolated']
starlink_diagnostics['err_u'] = starlink_diagnostics['starlink_u'] - starlink_diagnostics['gps_u_interpolated']
starlink_diagnostics['err_xy'] = np.hypot(starlink_diagnostics['err_e'], starlink_diagnostics['err_n'])
starlink_diagnostics['dt_sec'] = starlink_diagnostics['tr_corr'].diff()
starlink_diagnostics['starlink_step_xy_m'] = np.hypot(starlink_diagnostics['starlink_e'].diff(), starlink_diagnostics['starlink_n'].diff())
starlink_diagnostics['gps_step_xy_m'] = np.hypot(starlink_diagnostics['gps_e_interpolated'].diff(), starlink_diagnostics['gps_n_interpolated'].diff())

In [ ]:
xy_limit, xy_median, xy_mad = robust_upper_limit(starlink_diagnostics['err_xy'].to_numpy(dtype=float), STARLINK_OUTLIER_FLOOR_M)
u_limit, abs_u_median, abs_u_mad = robust_upper_limit(starlink_diagnostics['err_u'].abs().to_numpy(dtype=float), STARLINK_ALTITUDE_OUTLIER_FLOOR_M)
starlink_diagnostics['stale_flag'] = (
    (starlink_diagnostics['starlink_step_xy_m'] <= STARLINK_STALE_MAX_STEP_M)
    & (starlink_diagnostics['gps_step_xy_m'] >= REFERENCE_MOVEMENT_FOR_STALE_M)
)
starlink_diagnostics['xy_outlier_flag'] = starlink_diagnostics['err_xy'] > xy_limit
starlink_diagnostics['altitude_outlier_flag'] = starlink_diagnostics['err_u'].abs() > u_limit
flag_columns = ['stale_flag', 'xy_outlier_flag', 'altitude_outlier_flag']
starlink_diagnostics['flagged'] = starlink_diagnostics[flag_columns].any(axis=1)
starlink_diagnostics['issue'] = starlink_diagnostics.apply(
    lambda row: '+'.join([
        label for label, flag in (
            ('stale', row['stale_flag']),
            ('xy_outlier', row['xy_outlier_flag']),
            ('altitude_outlier', row['altitude_outlier_flag']),
        ) if bool(flag)
    ]),
    axis=1,
)
starlink_metrics = pd.DataFrame([{
    'scope': 'corrected_starlink_vs_dense_gps',
    'matching_mode': INTERPOLATED_MODE,
    'n': int(len(starlink_diagnostics)),
    'rmse_xy_m': float(np.sqrt(np.mean(starlink_diagnostics['err_xy'] ** 2))),
    'mae_xy_m': float(starlink_diagnostics['err_xy'].mean()),
    'p95_xy_m': float(starlink_diagnostics['err_xy'].quantile(0.95)),
    'max_xy_m': float(starlink_diagnostics['err_xy'].max()),
    'bias_u_m': float(starlink_diagnostics['err_u'].mean()),
    'rmse_u_m': float(np.sqrt(np.mean(starlink_diagnostics['err_u'] ** 2))),
    'p95_abs_u_m': float(starlink_diagnostics['err_u'].abs().quantile(0.95)),
    'stale_count': int(starlink_diagnostics['stale_flag'].sum()),
    'xy_outlier_count': int(starlink_diagnostics['xy_outlier_flag'].sum()),
    'altitude_outlier_count': int(starlink_diagnostics['altitude_outlier_flag'].sum()),
    'xy_outlier_threshold_m': float(xy_limit),
    'altitude_outlier_threshold_m': float(u_limit),
    'stale_starlink_max_step_m': STARLINK_STALE_MAX_STEP_M,
    'stale_reference_min_step_m': REFERENCE_MOVEMENT_FOR_STALE_M,
}])
flagged_starlink = starlink_diagnostics.loc[starlink_diagnostics['flagged']].copy()
starlink_diagnostics.to_csv(DIAGNOSTICS_DIR / 'starlink_vs_dense_gps.csv', index=False)
flagged_starlink.to_csv(DIAGNOSTICS_DIR / 'starlink_flagged_fixes.csv', index=False)
starlink_metrics.to_csv(DIAGNOSTICS_DIR / 'starlink_metrics.csv', index=False)
display(starlink_metrics)
starlink_display_columns = ['tr_corr', 'issue', 'err_xy', 'err_u', 'starlink_step_xy_m', 'gps_step_xy_m']
display(flagged_starlink.sort_values('err_xy', ascending=False).head(20)[starlink_display_columns])

In [ ]:
TIME_BIN_SEC = 60.0
default_bin_origin = min(aligned_interpolated.tr.min(), aligned_causal.tr.min())
BIN_ORIGIN_SEC = float(summary.get('active_t_start', default_bin_origin))


def errors_by_time_bin(table: pd.DataFrame, mode: str) -> pd.DataFrame:
    work = table.copy()
    work['bin_id'] = np.floor((work['tr'] - BIN_ORIGIN_SEC) / TIME_BIN_SEC).astype(int)
    rows: list[dict[str, Any]] = []
    for bin_id, group in work.groupby('bin_id', sort=True):
        metrics = summarize_position_errors(group, mode)
        metrics.update({
            'bin_id': int(bin_id),
            'bin_start_sec': float(BIN_ORIGIN_SEC + int(bin_id) * TIME_BIN_SEC),
            'bin_end_sec': float(BIN_ORIGIN_SEC + (int(bin_id) + 1) * TIME_BIN_SEC),
        })
        rows.append(metrics)
    return pd.DataFrame(rows)


time_bin_errors = pd.concat([
    errors_by_time_bin(aligned_interpolated, INTERPOLATED_MODE),
    errors_by_time_bin(aligned_causal, CAUSAL_MODE),
], ignore_index=True)
time_bin_errors.to_csv(DIAGNOSTICS_DIR / 'error_by_time_bin.csv', index=False)
time_bin_display_columns = ['matching_mode', 'bin_start_sec', 'bin_end_sec', 'n', 'rmse_xy_m', 'rmse_u_m', 'max_xy_m', 'max_abs_u_m']
display(time_bin_errors.sort_values('rmse_xy_m', ascending=False).head(12)[time_bin_display_columns])

## Vision coverage і factor counters

`used_vision_velocity` у summary — кількість створених factors, а `discarded_vision_velocity` — відкинуті samples. Це різні одиниці, тому з них не можна рахувати acceptance rate. Покриття нижче реконструюється з timestamps; точне зіставлення factors потребує окремого factor log.

In [ ]:
vision_work = vision.copy()
vision_work['speed_frd_mps'] = np.sqrt(vision_work['vx'] ** 2 + vision_work['vy'] ** 2 + vision_work['vz'] ** 2)
max_vision_speed = float(summary.get('max_vision_vel_frd_mps', np.inf))
speed_valid = np.isfinite(vision_work['speed_frd_mps']) & (vision_work['speed_frd_mps'] <= max_vision_speed)
active_start = float(summary.get('active_t_start', estimate['tr'].min()))
active_end = float(summary.get('active_t_end', estimate['tr'].max()))
active_mask = (vision_work['tr_corr'] >= active_start) & (vision_work['tr_corr'] <= active_end)
vision_candidate_input = vision_work.loc[speed_valid & active_mask].copy().reset_index(drop=True)
node_times = estimate['tr'].to_numpy(dtype=float)
measurement_times = vision_candidate_input['tr_corr'].to_numpy(dtype=float)
target_positions = np.searchsorted(node_times, measurement_times, side='right') - 1
valid_target = target_positions >= 0
target_age = np.full(len(target_positions), np.nan)
target_age[valid_target] = measurement_times[valid_target] - node_times[target_positions[valid_target]]
target_max_dt = float(summary.get('vision_target_max_dt_sec', np.inf))
candidate_factor_sample = valid_target & (target_age >= 0.0) & (target_age <= target_max_dt)
candidate_targets = target_positions[candidate_factor_sample]
unique_candidate_targets, samples_per_target = np.unique(candidate_targets, return_counts=True)
vision_dt = np.diff(vision_work['tr_corr'].to_numpy(dtype=float))
vision_dt = vision_dt[np.isfinite(vision_dt) & (vision_dt > 0)]
used_vision_factors = int(summary.get('used_vision_velocity', 0))
discarded_vision_samples = int(summary.get('discarded_vision_velocity', 0))
saved_nodes = int(summary.get('nodes_saved', len(estimate)))

In [ ]:
vision_coverage = pd.DataFrame([{
    'vision_samples_in_corrected_csv': int(len(vision_work)),
    'vision_samples_with_valid_speed': int(speed_valid.sum()),
    'vision_samples_in_active_window_and_speed_valid': int(len(vision_candidate_input)),
    'candidate_samples_matching_previous_node_within_max_dt': int(candidate_factor_sample.sum()),
    'unique_candidate_target_nodes': int(len(unique_candidate_targets)),
    'saved_graph_nodes': saved_nodes,
    'reconstructed_candidate_node_coverage_fraction': float(len(unique_candidate_targets) / saved_nodes) if saved_nodes else None,
    'median_candidate_samples_per_covered_node': float(np.median(samples_per_target)) if len(samples_per_target) else None,
    'p95_candidate_samples_per_covered_node': float(np.percentile(samples_per_target, 95)) if len(samples_per_target) else None,
    'median_source_period_sec': float(np.median(vision_dt)) if len(vision_dt) else None,
    'used_vision_factors_reported': used_vision_factors,
    'discarded_vision_samples_reported': discarded_vision_samples,
    'factor_count_minus_saved_nodes': used_vision_factors - saved_nodes,
    'factor_count_minus_reconstructed_unique_targets': used_vision_factors - int(len(unique_candidate_targets)),
    'vision_target_max_dt_sec': target_max_dt,
    'counter_units_note': 'used=factors; discarded=samples; do not combine into an acceptance ratio',
    'coverage_limit_note': 'timestamp reconstruction only; exact factor targets require a persisted factor log',
}])
vision_coverage.to_csv(DIAGNOSTICS_DIR / 'vision_coverage_and_factor_counts.csv', index=False)
display(vision_coverage.T)

In [ ]:
def best_scan_row(frame: pd.DataFrame, delay_column: str, score_column: str) -> tuple[pd.Series, bool]:
    clean = frame.copy()
    clean[delay_column] = pd.to_numeric(clean[delay_column], errors='coerce')
    clean[score_column] = pd.to_numeric(clean[score_column], errors='coerce')
    clean = clean.dropna(subset=[delay_column, score_column]).reset_index(drop=True)
    assert len(clean), f'No finite rows in scan columns {delay_column}, {score_column}'
    best_position = int(clean[score_column].to_numpy(dtype=float).argmin())
    return clean.iloc[best_position], best_position in (0, len(clean) - 1)


gps_scan_best, gps_scan_at_edge = best_scan_row(gps_imu_scan, 'gps_delay_sec', 'median_abs_course_yaw_error_deg')
starlink_scan_best, starlink_scan_at_edge = best_scan_row(starlink_scan, 'delay_sec', 'score_h_median')
timing_diagnostics = pd.DataFrame([
    {
        'signal': 'gps_to_imu',
        'selected_delay_sec': calibration.get('gps_delay_to_imu_sec'),
        'selected_delay_reliable': calibration.get('gps_delay_reliable'),
        'selection_reason': calibration.get('gps_delay_reason'),
        'best_scan_delay_sec': float(gps_scan_best['gps_delay_sec']),
        'best_scan_score': float(gps_scan_best['median_abs_course_yaw_error_deg']),
        'score_name': 'median_abs_course_yaw_error_deg',
        'best_scan_is_boundary': bool(gps_scan_at_edge),
    },
    {
        'signal': 'starlink_to_corrected_gps',
        'selected_delay_sec': calibration.get('starlink_delay_to_corrected_gps_sec'),
        'selected_delay_reliable': calibration.get('starlink_delay_reliable'),
        'selection_reason': 'calibration.json',
        'best_scan_delay_sec': float(starlink_scan_best['delay_sec']),
        'best_scan_score': float(starlink_scan_best['score_h_median']),
        'score_name': 'score_h_median_m',
        'best_scan_is_boundary': bool(starlink_scan_at_edge),
    },
    {
        'signal': 'vision_velocity_to_common_timebase',
        'selected_delay_sec': calibration.get('vision_velocity_delay_sec'),
        'selected_delay_reliable': None,
        'selection_reason': 'fixed parameter; no scan persisted',
        'best_scan_delay_sec': None,
        'best_scan_score': None,
        'score_name': None,
        'best_scan_is_boundary': None,
    },
])
timing_diagnostics.to_csv(DIAGNOSTICS_DIR / 'timing_diagnostics.csv', index=False)
display(timing_diagnostics)

In [ ]:
warnings_rows: list[dict[str, str]] = []


def add_warning(severity: str, code: str, evidence: str, implication: str) -> None:
    warnings_rows.append({'severity': severity, 'code': code, 'evidence': evidence, 'implication': implication})


gps_delay_selected = calibration.get('gps_delay_to_imu_sec')
gps_delay_reason = calibration.get('gps_delay_reason')
gps_scan_delay = float(gps_scan_best['gps_delay_sec'])
if not bool(calibration.get('gps_delay_reliable', False)):
    evidence = f'selected={gps_delay_selected} s; reason={gps_delay_reason}; scan_best={gps_scan_delay:.3f} s; scan_edge={gps_scan_at_edge}'
    add_warning(
        'warning', 'gps_imu_delay_unreliable', evidence,
        'GPS/IMU timing and/or yaw convention is not identified reliably; do not tune fusion noise around this uncertainty yet.',
    )

offset_rmse_h = float(calibration.get('offset_fit_rmseh_m', np.nan))
starlink_xy_sigma = float(summary.get('starlink_xy_sigma_m', np.nan))
if np.isfinite(offset_rmse_h) and np.isfinite(starlink_xy_sigma) and offset_rmse_h > 3.0 * starlink_xy_sigma:
    evidence = f'offset fit horizontal RMSE={offset_rmse_h:.2f} m; factor sigma={starlink_xy_sigma:.2f} m'
    add_warning(
        'warning', 'starlink_fit_vs_factor_sigma', evidence,
        'The position factor is much more confident than the calibration residual supports.',
    )

summary_v0 = np.asarray(summary.get('v0_enu_mps', [0.0, 0.0, 0.0]), dtype=float)
calibration_v0 = np.asarray(calibration.get('v0_enu_mps', [0.0, 0.0, 0.0]), dtype=float)
v0_disagreement = float(np.linalg.norm(summary_v0 - calibration_v0))
if v0_disagreement > 1.0:
    evidence = f'graph v0={summary_v0.tolist()} m/s; calibration v0={calibration_v0.tolist()} m/s; difference={v0_disagreement:.2f} m/s'
    add_warning(
        'warning', 'initial_velocity_disagreement', evidence,
        'Initialization depends strongly on which velocity estimate is selected.',
    )

p0_source = str(summary.get('p0_source', ''))
v0_source = str(summary.get('v0_source', ''))
if 'gps1' in p0_source or 'gps1' in v0_source:
    add_warning(
        'info', 'dense_gps_used_for_initialization', f'p0_source={p0_source}; v0_source={v0_source}',
        'This run is a benchmark prototype, not a deployment-pure sparse-GPS run.',
    )

In [ ]:
factor_excess = used_vision_factors - saved_nodes
if factor_excess > 0:
    evidence = f'reported factors={used_vision_factors}; saved nodes={saved_nodes}; excess={factor_excess}'
    add_warning(
        'warning', 'vision_factor_count_exceeds_nodes', evidence,
        'Some graph nodes likely received more than one vision factor; persist a factor-target log before changing aggregation.',
    )

xy_outlier_count = int(starlink_diagnostics['xy_outlier_flag'].sum())
if xy_outlier_count and not bool(summary.get('robust_starlink_position_noise', False)):
    evidence = f'diagnostic XY outliers={xy_outlier_count}; robust noise=False; threshold={xy_limit:.2f} m'
    add_warning(
        'warning', 'starlink_outliers_without_robust_noise', evidence,
        'Large sparse fixes can pull the graph disproportionately.',
    )

stale_count = int(starlink_diagnostics['stale_flag'].sum())
if stale_count:
    evidence = f'stale intervals={stale_count}; Starlink step <= {STARLINK_STALE_MAX_STEP_M} m while GPS step >= {REFERENCE_MOVEMENT_FOR_STALE_M} m'
    add_warning(
        'warning', 'stale_starlink_fixes', evidence,
        'Timestamp-correct but stale coordinates should be detected before they become graph priors.',
    )

exported_alt_interp = altitude_metrics.loc[
    (altitude_metrics['series'] == 'fusion_exported_u')
    & (altitude_metrics['matching_mode'] == INTERPOLATED_MODE)
].iloc[0]
baro_sigma = float(summary.get('baro_z_sigma_m', np.nan))
exported_alt_rmse = float(exported_alt_interp['rmse_u_m'])
if np.isfinite(baro_sigma) and exported_alt_rmse > 3.0 * baro_sigma:
    evidence = f'exported altitude RMSE={exported_alt_rmse:.2f} m; barometer factor sigma={baro_sigma:.2f} m'
    add_warning(
        'warning', 'altitude_error_exceeds_baro_sigma', evidence,
        'Vertical error is too large to explain only by the configured barometer noise.',
    )

failed_updates = int(summary.get('failed_updates', 0))
if failed_updates > 0:
    add_warning('error', 'gtsam_failed_updates', f'failed_updates={failed_updates}', 'The baseline did not complete cleanly.')

diagnostic_warnings = pd.DataFrame(warnings_rows, columns=['severity', 'code', 'evidence', 'implication'])
diagnostic_warnings.to_csv(DIAGNOSTICS_DIR / 'diagnostic_warnings.csv', index=False)
display(diagnostic_warnings)

In [ ]:
plot_stride = max(1, len(aligned_interpolated) // 4000)
plot_alignment = aligned_interpolated.iloc[::plot_stride]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(gps['e'], gps['n'], '.', ms=1, alpha=0.35, label='dense GPS reference')
axes[0].plot(estimate['e'], estimate['n'], lw=1.2, label='synthetic GPS')
axes[0].plot(starlink['e_corr'], starlink['n_corr'], 'o', ms=3, alpha=0.55, label='corrected Starlink')
axes[0].set_aspect('equal', adjustable='box')
axes[0].set_xlabel('E, m')
axes[0].set_ylabel('N, m')
axes[0].set_title('Trajectory')
axes[0].grid(True)
axes[0].legend()
axes[1].plot(plot_alignment['tr'], plot_alignment['err_xy'], label='horizontal error')
axes[1].plot(plot_alignment['tr'], plot_alignment['abs_err_u'], label='absolute altitude error')
axes[1].set_xlabel('relative time, s')
axes[1].set_ylabel('error, m')
axes[1].set_title('Offline interpolated errors')
axes[1].grid(True)
axes[1].legend()
fig.tight_layout()
fig.savefig(DIAGNOSTICS_DIR / 'trajectory_and_errors.png', dpi=140, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(gps['tr'], gps['u'], '.', ms=1.5, alpha=0.4, label='dense GPS u')
if 'u_raw' in estimate.columns:
    axes[0].plot(estimate['tr'], estimate['u_raw'], lw=0.8, alpha=0.55, label='fusion raw u')
axes[0].plot(estimate['tr'], estimate['u'], lw=1.2, label='fusion exported u')
axes[0].set_ylabel('U, m')
axes[0].set_title('Altitude series')
axes[0].grid(True)
axes[0].legend()
axes[1].plot(plot_alignment['tr'], plot_alignment['err_u'], label='exported u - dense GPS u')
axes[1].axhline(0.0, color='black', lw=0.8)
axes[1].set_xlabel('relative time, s')
axes[1].set_ylabel('signed error, m')
axes[1].grid(True)
axes[1].legend()
fig.tight_layout()
fig.savefig(DIAGNOSTICS_DIR / 'altitude_diagnostics.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=False)
axes[0].plot(starlink_diagnostics['tr_corr'], starlink_diagnostics['err_xy'], 'o-', ms=3, lw=0.8, label='Starlink XY error')
axes[0].axhline(xy_limit, color='tab:red', linestyle='--', label=f'outlier threshold {xy_limit:.1f} m')
flagged_xy = starlink_diagnostics['stale_flag'] | starlink_diagnostics['xy_outlier_flag']
axes[0].scatter(starlink_diagnostics.loc[flagged_xy, 'tr_corr'], starlink_diagnostics.loc[flagged_xy, 'err_xy'], color='tab:red', zorder=3, label='stale/outlier')
axes[0].set_xlabel('relative time, s')
axes[0].set_ylabel('XY error, m')
axes[0].set_title('Corrected Starlink vs dense GPS')
axes[0].grid(True)
axes[0].legend()
for mode, group in time_bin_errors.groupby('matching_mode'):
    axes[1].plot(group['bin_start_sec'], group['rmse_xy_m'], 'o-', label=f'XY: {mode}')
    axes[1].plot(group['bin_start_sec'], group['rmse_u_m'], '.--', label=f'U: {mode}')
axes[1].set_xlabel('time-bin start, s')
axes[1].set_ylabel('RMSE, m')
axes[1].set_title(f'Errors by {TIME_BIN_SEC:.0f}-second bin')
axes[1].grid(True)
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(DIAGNOSTICS_DIR / 'starlink_and_time_bins.png', dpi=140, bbox_inches='tight')
plt.show()

In [ ]:
def records_for_json(frame: pd.DataFrame) -> list[dict[str, Any]]:
    return json.loads(frame.to_json(orient='records'))


diagnostic_metrics = pd.concat([position_metrics, starlink_metrics, altitude_metrics], ignore_index=True, sort=False)
diagnostic_metrics.to_csv(DIAGNOSTICS_DIR / 'diagnostic_metrics.csv', index=False)
generated_files = [
    'diagnostic_metrics.csv',
    'diagnostics_summary.json',
    'output_vs_dense_gps_metrics.csv',
    'output_vs_dense_gps_interpolated.csv',
    'output_vs_dense_gps_causal_previous.csv',
    'altitude_metrics.csv',
    'altitude_alignment_details.csv',
    'starlink_metrics.csv',
    'starlink_vs_dense_gps.csv',
    'starlink_flagged_fixes.csv',
    'error_by_time_bin.csv',
    'vision_coverage_and_factor_counts.csv',
    'timing_diagnostics.csv',
    'diagnostic_warnings.csv',
    'trajectory_and_errors.png',
    'altitude_diagnostics.png',
    'starlink_and_time_bins.png',
]

In [ ]:
diagnostics_summary: dict[str, Any] = {
    'schema_version': 1,
    'generated_utc': datetime.now(timezone.utc).isoformat(),
    'contract': {
        'gtsam_rerun': False,
        'algorithm_modified': False,
        'input_artifacts_read_only': True,
        'dense_gps_role': 'diagnostic_reference_only',
    },
    'inputs': {
        'synthetic_gps': str(OUTPUT_PATH),
        'fusion_summary': str(SUMMARY_PATH),
        'calibration': str(CALIBRATION_PATH),
        'dense_gps': str(GPS_PATH),
        'corrected_starlink': str(STARLINK_PATH),
        'corrected_vision_velocity': str(VISION_PATH),
    },
    'matching_modes': {
        INTERPOLATED_MODE: 'offline linear interpolation between adjacent output nodes',
        CAUSAL_MODE: 'latest output node at or before reference time; zero-order hold',
    },
    'position_metrics': records_for_json(position_metrics),
    'altitude_metrics': records_for_json(altitude_metrics),
    'starlink_metrics': records_for_json(starlink_metrics),
    'vision_coverage': records_for_json(vision_coverage),
    'timing_diagnostics': records_for_json(timing_diagnostics),
    'warnings': records_for_json(diagnostic_warnings),
    'time_bins': {
        'bin_size_sec': TIME_BIN_SEC,
        'rows': int(len(time_bin_errors)),
        'details_file': 'error_by_time_bin.csv',
    },
    'generated_files': generated_files,
}
summary_output_path = DIAGNOSTICS_DIR / 'diagnostics_summary.json'
summary_output_path.write_text(
    json.dumps(diagnostics_summary, indent=2, ensure_ascii=False, allow_nan=False),
    encoding='utf-8',
)
print('Diagnostics completed without rerunning GTSAM.')
print('Summary:', summary_output_path)
print('Generated files:')
for filename in generated_files:
    path = DIAGNOSTICS_DIR / filename
    print(f'  {filename:48s} {path.stat().st_size:12d} bytes')